# MJBoost 체험: 2차를 넘는 상호작용까지 해석되는 부스팅

**MJBoost**는 라운드마다 Kolmogorov–Arnold 유닛 $g\big(\phi_1(x_1)+\phi_2(x_2)+\cdots\big)$ 하나를 더하는 그래디언트 부스팅입니다.
안쪽 함수 $\phi$와 바깥 함수 $g$는 벌점 스플라인이고, 각 유닛은 닫힌 형태로 맞춥니다.
유닛이 각 입력을 안쪽 함수 하나로만 보기 때문에, 학습된 모델은 **주효과 + 쌍효과 + 3차 이상 나머지**로 *정확히* 분해됩니다.

이 노트북은 다섯 가지를 보여줍니다.

1. **합성 자료**: 세 입력이 함께 작용하는 목표를 MJBoost는 배우고, 쌍까지만 보는 EBM은 못 배웁니다.
2. **캘리포니아 집값**: 학습된 모델을 곡선, 위치 지도, 분산 지분으로 읽습니다.
3. **펭귄**: 범주형 입력과 결측값을 전처리 없이 그대로 넣습니다.
4. **유방암 분류**: 이진 분류와 로그 오즈 해석.
5. **펭귄 종 분류**: 다중분류(소프트맥스)와 클래스별 해석.

**실행:** 메뉴 *런타임 → 모두 실행*. 기본 CPU 런타임(코어 2개)에서 전체 15분 안팎 걸립니다. 가장 오래 걸리는 칸은 캘리포니아 집값(MJBoost와 EBM 각각 몇 분)입니다.
첫 학습은 numba 커널을 컴파일하느라 수십 초가 더 걸립니다.

In [ ]:
# @title MJBoost 설치 { display-mode: "form" }
# PyPI의 mjboost(https://pypi.org/project/mjboost/)와 비교용 EBM(interpret-core)을 설치합니다.
# LightGBM은 Colab에 이미 있습니다.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mjboost>=0.2.0", "interpret-core"], check=True)

import mjboost
print("mjboost", mjboost.__version__, "ready")

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from mjboost import MJBoostRegressor, MJBoostClassifier

warnings.filterwarnings("ignore")
N_JOBS = min(2, os.cpu_count() or 1)          # Colab 기본 런타임은 CPU 2개
try:
    from interpret.glassbox import ExplainableBoostingRegressor, ExplainableBoostingClassifier
    HAVE_EBM = True
except ImportError:
    HAVE_EBM = False
try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    HAVE_LGB = False

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(f"CPU {os.cpu_count()}, EBM {'O' if HAVE_EBM else 'X'}, LightGBM {'O' if HAVE_LGB else 'X'}")

## 1. 합성 자료: 세 입력이 함께 작용하는 목표

입력 8개를 $U(-1.5, 1.5)$에서 뽑고, 앞의 3개만 목표에 쓰입니다(나머지 5개는 잡음 입력).

$$f(x) = \exp(s),\qquad s = \tfrac{1}{\sqrt3}\sum_{p\le 3}\big(x_p + 0.5\sin 2x_p\big)$$

$f$의 분산 중 약 3 %는 세 입력이 동시에 만드는 **3차 상호작용**입니다.
주효과와 쌍효과만으로는 이 부분을 표현할 수 없습니다.
학습 3,000행, 시험 2,000행, 잡음 표준편차는 $0.1\,\mathrm{sd}(f)$이고, 오차는 잡음 없는 $f$에 대해 잽니다.

In [ ]:
rng = np.random.default_rng(3)
D, K = 8, 3
X = rng.uniform(-1.5, 1.5, (5000, D))
s = (X[:, :K] + 0.5 * np.sin(2 * X[:, :K])).sum(axis=1) / np.sqrt(K)
f = np.exp(s)
y = f + 0.1 * f.std() * rng.standard_normal(len(f))
cols = [f"x{j + 1}" for j in range(D)]
Xtr_s, Xte_s = pd.DataFrame(X[:3000], columns=cols), pd.DataFrame(X[3000:], columns=cols)
ytr_s, fte_s = y[:3000], f[3000:]

fits = {}
t = time.time()
mj_syn = MJBoostRegressor(n_jobs=N_JOBS).fit(Xtr_s, ytr_s)
fits["MJBoost"] = (mj_syn.predict(Xte_s), time.time() - t)
print(f"MJBoost: {time.time() - t:.0f} s, units {mj_syn.n_units_}, selection {mj_syn.selection_['race']}")

if HAVE_EBM:
    t = time.time()
    ebm = ExplainableBoostingRegressor(n_jobs=N_JOBS, random_state=0).fit(Xtr_s, ytr_s)
    fits["EBM (pairs)"] = (ebm.predict(Xte_s), time.time() - t)
if HAVE_LGB:
    t = time.time()
    lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, n_jobs=N_JOBS, verbose=-1).fit(Xtr_s, ytr_s)
    fits["LightGBM"] = (lg.predict(Xte_s), time.time() - t)

# MJBoost truncated to its main and pair effects (its own order-2 part)
ex_syn = mj_syn.explain()
C = ex_syn.components(Xte_s)
upto2 = C["f0"] + C["main"].sum(axis=1).to_numpy() + sum(C["pair"].values(), np.zeros(len(Xte_s)))
fits["MJBoost, main + pairs only"] = (upto2, float("nan"))

pd.DataFrame({name: {"normalized RMSE (vs noise-free f)": rmse(p, fte_s) / fte_s.std(), "fit seconds": sec}
              for name, (p, sec) in fits.items()}).T.round(3)

**읽는 법:** 값이 작을수록 좋습니다(0이면 완벽, 1이면 평균만 맞히는 수준).
쌍까지만 보는 EBM과, MJBoost에서 주효과·쌍효과만 남긴 모델은 3차 상호작용을 놓쳐서 오차가 비슷하게 큽니다.
MJBoost 전체 모델은 3차 이상 부분까지 담아서 오차가 훨씬 작습니다.

이번에는 분해가 **참값**을 얼마나 되찾는지 봅니다.
이 목표는 독립 입력의 곱 꼴이라, 차수별 분산 지분의 참값을 닫힌 형태로 계산할 수 있습니다.

In [ ]:
from math import comb

def true_shares(k):
    """Population variance shares of the order-j parts of f = exp(s) (product of independent factors)."""
    t, w = np.polynomial.legendre.leggauss(400)
    x, w = 1.5 * t, w / 2.0
    z = np.exp((x + 0.5 * np.sin(2 * x)) / np.sqrt(k))
    mu, v = w @ z, w @ (z - w @ z) ** 2
    V = np.array([comb(k, j) * v ** j * mu ** (2 * (k - j)) for j in range(1, k + 1)])
    V = V / V.sum()
    return V[0], V[1], V[2:].sum()

sh = ex_syn.shares(Xte_s)
tab = pd.DataFrame({"MJBoost decomposition": [sh["main"], sh["pairs"], sh["higher_order"]],
                    "true": true_shares(K)}, index=["main effects", "pairs", "order >= 3"]).round(3)
display(tab)
tab.plot.bar(rot=0, figsize=(5, 2.6), color=["#D55E00", "#444444"]); plt.ylabel("variance share"); plt.show()
print(ex_syn.importance(Xte_s).head(8).round(3).to_string(index=False))

잡음 입력 `x4`–`x8`은 중요도 표의 아래쪽에 거의 0으로 남습니다.
주효과·쌍효과·3차 이상 지분이 참값과 거의 같습니다.

## 2. 캘리포니아 집값: 모델 읽기

1990년 캘리포니아 인구조사 구역 20,640개의 중위 집값(단위 10만 달러), 입력 8개입니다.
기본 CPU 런타임에서 학습에 몇 분 걸립니다.

In [ ]:
from sklearn.datasets import fetch_california_housing
cal = fetch_california_housing(as_frame=True)
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(cal.data, cal.target, test_size=0.2, random_state=0)

scores = {}
t = time.time()
mj_cal = MJBoostRegressor(n_jobs=N_JOBS).fit(Xtr_c, ytr_c)
scores["MJBoost"] = (rmse(mj_cal.predict(Xte_c), yte_c), time.time() - t)
print(f"MJBoost: {time.time() - t:.0f} s, units {mj_cal.n_units_}, chosen {mj_cal.selection_['candidate']}")
if HAVE_EBM:
    t = time.time()
    ebm_c = ExplainableBoostingRegressor(n_jobs=N_JOBS, random_state=0).fit(Xtr_c, ytr_c)
    scores["EBM"] = (rmse(ebm_c.predict(Xte_c), yte_c), time.time() - t)
if HAVE_LGB:
    t = time.time()
    lg_c = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=N_JOBS, verbose=-1).fit(Xtr_c, ytr_c)
    scores["LightGBM"] = (rmse(lg_c.predict(Xte_c), yte_c), time.time() - t)
pd.DataFrame(scores, index=["test RMSE (100k USD)", "fit seconds"]).T.round(3)

In [ ]:
ex_cal = mj_cal.explain()
sh = ex_cal.shares(Xte_c, yte_c)
print(f"variance shares: main {sh['main']:.3f}, pairs {sh['pairs']:.3f}, order>=3 {sh['higher_order']:.3f}")
print(f"test RMSE: full {sh['rmse_full']:.3f}, main + pairs {sh['rmse_main_and_pairs']:.3f}, main only {sh['rmse_main']:.3f}")
ex_cal.importance(Xte_c).head(10).round(3)

**주효과 곡선**: 다른 입력을 평균낸 각 입력의 효과입니다(단위 10만 달러, 회색 점선이 0).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
for ax, name in zip(axes, ["MedInc", "HouseAge", "AveRooms", "AveOccup"]):
    ex_cal.plot_main(name, ax=ax)
plt.tight_layout(); plt.show()

**위치 효과 지도**: 위도·경도의 주효과와 위도×경도 쌍효과를 더하면 각 구역의 위치 효과가 됩니다.
해안(샌프란시스코만, 로스앤젤레스)이 높고 내륙이 낮게 나와야 합니다.

In [ ]:
C = ex_cal.components(Xte_c)
pair = C["pair"].get(("Latitude", "Longitude"), C["pair"].get(("Longitude", "Latitude"), 0.0))
loc = C["main"]["Latitude"] + C["main"]["Longitude"] + pair
lim = np.quantile(np.abs(loc - np.median(loc)), 0.99)
fig, ax = plt.subplots(figsize=(5.2, 4.8))
sc = ax.scatter(Xte_c["Longitude"], Xte_c["Latitude"], c=loc - np.median(loc), s=4, cmap="RdBu_r", vmin=-lim, vmax=lim)
plt.colorbar(sc, ax=ax, label="location effect vs. median (100k USD)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_title("Location effect of each test district")
plt.show()

## 3. 범주형 입력과 결측값: 펭귄 몸무게

`species`, `island`, `sex`는 문자열 열이라 자동으로 범주형이 됩니다. 결측값(NaN)도 그대로 넣습니다.
범주형 입력은 유닛 안에서 입력 하나로 들어갑니다(원-핫으로 쪼개지 않음).

In [ ]:
import seaborn as sns
peng = sns.load_dataset("penguins").dropna(subset=["body_mass_g"])
Xp, yp = peng.drop(columns="body_mass_g"), peng["body_mass_g"]
print("missing values per column:", Xp.isna().sum().to_dict())
Xtr_p, Xte_p, ytr_p, yte_p = train_test_split(Xp, yp, test_size=0.25, random_state=0)

mj_peng = MJBoostRegressor(n_jobs=N_JOBS).fit(Xtr_p, ytr_p)
print("categorical inputs:", mj_peng.prep_.cat_names_)
print(f"test RMSE: MJBoost {rmse(mj_peng.predict(Xte_p), yte_p):.0f} g, mean only {rmse(np.full(len(yte_p), ytr_p.mean()), yte_p):.0f} g")

In [ ]:
ex_p = mj_peng.explain()
display(ex_p.main_effect("species").round(1))
display(ex_p.main_effect("sex").round(1))     # value None = missing sex
new = pd.DataFrame([{"species": "Gentoo", "island": "Unknown island", "bill_length_mm": 47.0, "bill_depth_mm": np.nan,
                     "flipper_length_mm": 215.0, "sex": np.nan}])
print("prediction for a penguin with an unseen island and missing values:", mj_peng.predict(new).round(0), "g")

## 4. 이진 분류: 유방암 진단

`MJBoostClassifier`는 같은 유닛을 뉴턴 단계로 맞춥니다. 분해와 효과는 **로그 오즈** 척도입니다.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
Xb, yb = load_breast_cancer(return_X_y=True, as_frame=True)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)

probs = {}
t = time.time()
clf = MJBoostClassifier(n_jobs=N_JOBS).fit(Xtr_b, ytr_b)
probs["MJBoost"] = (clf.predict_proba(Xte_b)[:, 1], time.time() - t)
if HAVE_EBM:
    t = time.time()
    e = ExplainableBoostingClassifier(n_jobs=N_JOBS, random_state=0).fit(Xtr_b, ytr_b)
    probs["EBM"] = (e.predict_proba(Xte_b)[:, 1], time.time() - t)
if HAVE_LGB:
    t = time.time()
    g = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, n_jobs=N_JOBS, verbose=-1).fit(Xtr_b, ytr_b)
    probs["LightGBM"] = (g.predict_proba(Xte_b)[:, 1], time.time() - t)
pd.DataFrame({k: {"accuracy": accuracy_score(yte_b, p > 0.5), "AUC": roc_auc_score(yte_b, p),
                  "log-loss": log_loss(yte_b, np.clip(p, 1e-15, 1 - 1e-15)), "fit seconds": sec}
              for k, (p, sec) in probs.items()}).T.round(4)

In [ ]:
ex_b = clf.explain()
top = ex_b.importance(Xte_b)
display(top.head(6).round(3))
mains = [t for t, k in zip(top["term"], top["kind"]) if k == "main"][:3]
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
for ax, name in zip(axes, mains):
    ex_b.plot_main(name, ax=ax)
plt.tight_layout(); plt.show()

## 5. 다중분류: 펭귄 종

클래스가 셋 이상이면 `MJBoostClassifier`가 자동으로 소프트맥스 부스팅을 합니다. 라운드마다 클래스별로 유닛 하나씩을 더합니다.
해석은 클래스 하나를 골라 그 클래스의 로짓을 분해합니다: `explain(class_="Gentoo")`.

In [ ]:
from sklearn.metrics import accuracy_score, log_loss
peng_s = sns.load_dataset("penguins").dropna(subset=["species"])
Xs, ys = peng_s.drop(columns="species"), peng_s["species"]
Xtr_m, Xte_m, ytr_m, yte_m = train_test_split(Xs, ys, test_size=0.25, random_state=0, stratify=ys)

res = {}
t = time.time()
mj_sp = MJBoostClassifier(n_jobs=N_JOBS).fit(Xtr_m, ytr_m)
res["MJBoost"] = (mj_sp.predict_proba(Xte_m), time.time() - t)
num = Xtr_m.select_dtypes("number").columns          # EBM and LightGBM: numeric columns, categories as codes
enc = lambda D: pd.concat([D[num], D.drop(columns=num).astype("category").apply(lambda c: c.cat.codes)], axis=1)
if HAVE_EBM:
    t = time.time()
    e = ExplainableBoostingClassifier(n_jobs=N_JOBS, random_state=0).fit(enc(Xtr_m), ytr_m)
    res["EBM"] = (e.predict_proba(enc(Xte_m)), time.time() - t)
if HAVE_LGB:
    t = time.time()
    g = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, n_jobs=N_JOBS, verbose=-1).fit(enc(Xtr_m), ytr_m)
    res["LightGBM"] = (g.predict_proba(enc(Xte_m)), time.time() - t)
print("classes:", list(mj_sp.classes_))
pd.DataFrame({k: {"accuracy": accuracy_score(yte_m, mj_sp.classes_[P.argmax(1)]),
                  "log-loss": log_loss(yte_m, np.clip(P, 1e-15, 1), labels=mj_sp.classes_), "fit seconds": sec}
              for k, (P, sec) in res.items()}).T.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, c in zip(axes, mj_sp.classes_):
    ex_c = mj_sp.explain(class_=c)
    ex_c.plot_main("flipper_length_mm", ax=ax)
    ax.set_title(c)
plt.tight_layout(); plt.show()
mj_sp.explain(class_="Gentoo").importance(Xte_m).head(6).round(3)

## 6. 저장과 불러오기

학습된 모델은 `joblib`이나 `pickle`로 저장하고 그대로 불러올 수 있습니다. scikit-learn의 `Pipeline`, `clone`, `cross_val_score`와도 함께 쓸 수 있습니다.

In [ ]:
import joblib
joblib.dump(mj_cal, "mjboost_california.joblib")
loaded = joblib.load("mjboost_california.joblib")
print("same predictions after reload:", np.array_equal(loaded.predict(Xte_c), mj_cal.predict(Xte_c)))
print(f"file size: {os.path.getsize('mjboost_california.joblib') / 1e6:.1f} MB")

## 옵션 요약

| 인자 | 기본값 | 뜻 |
|---|---|---|
| `selection` | `"two_race"` | 논문 설정: 가지치기 켠 8후보 경주와 16후보 경주를 같은 내부 fold에서 돌려 내부 CV가 낮은 쪽. `"pruned"`는 첫 경주만 써서 계산이 약 1/3 |
| `cat_features` | `"auto"` | 범주형 열: `"auto"`(문자열·category·bool 열), `None`, 또는 열 이름/위치 목록 |
| `n_jobs` | 1 | 내부 교차검증 병렬 프로세스 수(결과는 같음) |
| `max_units` | 4000 | fold 모델당 최대 유닛 수(조기 종료가 실제 개수를 정함) |
| `random_state` | 0 | 내부 fold와 열 추출의 시드 |

해석 API: `explain()` → `shares`, `importance`, `main_effect`, `pair_effect`, `components`, `plot_main`, `plot_pair`, `plot_shares`.